# 04 — Task Lifecycle: States, Polling, Cancel, and Multi-Turn

## Why this notebook exists

In **notebook 03** every `message/send` call returned a `Task` that was already `completed` (or `failed`). The agent did its work synchronously inside the request handler and the client got the answer in the same HTTP round-trip.

That's fine for fast lookups. Real agents do things that take seconds to minutes: hitting external APIs, running models, waiting on humans. A2A models that with a small but powerful state machine on the `Task` object itself, plus three methods that operate on long-running tasks:

- `tasks/get` — *"what's the status of task X right now?"*
- `tasks/cancel` — *"stop working on task X."*
- A second `message/send` with `taskId` set — *"here's the clarification you asked for on task X."*

This notebook builds a "slow researcher" that runs work in a background thread, exposes all four methods, and walks through every transition you can plausibly hit.

> *Targets A2A spec v0.3.0.*

## What you'll learn

- The A2A task state machine and which states are terminal vs. transient.
- How to implement a server that spawns background work and returns a `submitted` task immediately.
- How `tasks/get` lets a client poll for progress.
- How `tasks/cancel` stops in-flight work cleanly.
- How a server requests more information mid-task with the `input-required` state, and how the client replies with a follow-up `message/send` whose `Message.taskId` references the original task.
- Why polling is fundamentally wasteful — setting up notebook 05's streaming.

## 1. Setup

Same helpers as previous notebooks, plus `threading.Lock`, `threading.Event`, and a couple of helpers for the task store.

In [ ]:
import json
import threading
import time
import uuid
from datetime import datetime, timezone
from typing import Literal

import httpx
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field, ValidationError, model_validator

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")

    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


print("Setup OK")

## 2. The Task Lifecycle

A2A tasks carry a `status.state` that walks a deliberate state machine. Most calls move the task from one state to another; some states are terminal (the task is done, one way or another).

```
        ┌───────────┐
        │ submitted │  ← initial state on message/send acceptance
        └─────┬─────┘
              │ worker picks it up
              ▼
        ┌───────────┐                ┌────────────────┐
        │  working  │ ─ needs info ▶ │ input-required │
        └─────┬─────┘                └───────┬────────┘
              │                              │ client sends
              │                              │ message/send w/ taskId
              ▼                              ▼
   ┌──────────────────┐                ┌──────────┐
   │     completed    │ ◀──────────────│  working │
   └──────────────────┘  (resumed)     └─────┬────┘
                                             │
                          ┌──────────┐       │
                          │  failed  │ ◀─────┤
                          └──────────┘       │
                          ┌──────────┐       │
                          │ canceled │ ◀─────┘  (via tasks/cancel)
                          └──────────┘
```

Terminal states: **`completed`**, **`failed`**, **`canceled`**, **`rejected`**, **`auth-required`** (in the sense that further protocol action is needed before work continues).

Transient states: **`submitted`**, **`working`**, **`input-required`**, **`unknown`**.

The full v0.3.0 set: `submitted`, `working`, `input-required`, `completed`, `canceled`, `failed`, `rejected`, `auth-required`, `unknown`. (Notebook 03 already defined `TaskState` with all nine values.)

We'll exercise `submitted → working → completed`, `working → canceled`, and `working → input-required → working → completed` in this notebook.